# PASNet Output Analysis

This notebook loads a **trained PASNet model** (Pathway-Associated Sparse deep Neural Network) produced by the Nextflow pipeline, evaluates it on a held-out validation set, and explores *why* it makes the predictions it does.

**What this notebook does, step by step:**
1. Import the pipeline's custom utility code and third-party libraries
2. Set the clinical/molecular variable being predicted
3. Load the trained model checkpoint
4. Evaluate it on the validation cohort (AUC, F1)
5. Plot and save the ROC curve
6. Save raw model predictions to disk
7. Compute SHAP values to explain *which genes/pathways* drive predictions
8. Render a publication-style SHAP summary plot
9. Extract the first sparse layer's weights (the pathway -> hidden-unit connections) for downstream interpretation

> **Note:** All file paths are relative to the pipeline's `pipeline/output/pasnet/...` directory structure produced by the Nextflow run, so this notebook is expected to live alongside that `pipeline/` folder.


## 1. Environment Setup & Pipeline Utility Imports

Before anything else, we need access to the helper functions written for the PASNet Nextflow pipeline (`utils.data_utils`, `utils.metrics`, `utils.model`). These live in `pipeline/bin/utils/` relative to the pipeline root, **one level above this notebook's folder**.

Rather than hard-coding an absolute path (which would break on every other machine/user), we compute the utilities path dynamically from the notebook's own working directory, and print out diagnostic checks so that any path mismatch is obvious immediately rather than surfacing as a confusing `ModuleNotFoundError` several cells later.


In [ ]:
# --- Standard library & third-party imports -------------------------------
import sys
import os
from pathlib import Path
import pickle

import torch
import numpy as np
import pandas as pd
import shap

# --- Locate the pipeline's custom 'utils' package --------------------------
# The notebook lives in <pipeline_root>/PostHoc_Analyses (or similar), and the
# reusable helper code lives in <pipeline_root>/pipeline/bin/utils. We derive
# that path from the notebook's current working directory so this notebook is
# portable across machines/users instead of relying on a hard-coded path.
notebook_dir = Path(os.getcwd())
utils_path = notebook_dir.parent / "pipeline" / "bin"

# --- Diagnostics -------------------------------------------------------------
# Print everything we're about to rely on so a broken path is obvious at a
# glance, instead of surfacing as a cryptic ModuleNotFoundError later on.
print("🔍 Notebook Folder:", notebook_dir)
print("🔍 Calculated Target Utilities Folder:", utils_path)
print("📁 Does this utilities path actually exist?:", utils_path.exists())

# Make the utilities importable for the rest of this session
if str(utils_path) not in sys.path:
    sys.path.append(str(utils_path))

# Sanity-check that the package is actually a proper Python package
init_file = utils_path / "utils" / "__init__.py"
print("📄 Does 'utils/__init__.py' exist inside that path?:", init_file.exists())

# --- Import the actual PASNet pipeline utilities ----------------------------
# load_data      -> reads an Excel sheet of expression/clinical data into
#                   model-ready tensors
# load_pathway   -> loads the gene -> pathway membership mask used to build
#                   PASNet's sparse first layer
# calc_auc, f1   -> evaluation metrics matched to how PASNet was trained
# PASNet,
# trainPASNet    -> the model class itself and its training loop
try:
    from utils.data_utils import load_data, load_pathway
    from utils.metrics import calc_auc, f1, binary_cross_entropy_for_imbalance
    from utils.model import PASNet, trainPASNet
    print("\n✅ Success! All PASNet pipeline utilities imported cleanly.")
except ModuleNotFoundError as e:
    print(f"\n❌ Import Failed: {e}")
    print("Check your folder structure against the path listed above.")


## 2. Analysis Configuration

`comparison_col` names the clinical/molecular label column in the input spreadsheets that the model was trained to predict. Changing this single variable would let you re-run the same notebook against a model trained on a different outcome label, as long as the column exists in the data files below.


In [ ]:
# Name of the binary outcome column the model was trained to predict.
# This must exactly match a column header in the Training/Validation Excel files.
comparison_col = "."


## 3. Load the Trained Model

The Nextflow pipeline trains PASNet and serializes the final model object with `pickle`. We load that checkpoint here; it contains the full PyTorch `nn.Module` (architecture + learned weights), not just a state dictionary, so it can be used directly for inference.


In [ ]:
# Load the trained PASNet model object exactly as it was saved by the pipeline.
model_path = "../pipeline/output/pasnet/PASNet/Output/GO/model.pkl"

with open(model_path, "rb") as file:
    model = pickle.load(file)


## 4. Evaluate Model Performance on the Validation Set

With the model loaded, we:
1. Switch it to **evaluation mode** (`model.eval()`), which disables training-only behaviours such as dropout.
2. Load both the **training** set (needed later as a SHAP background reference) and the **validation** set, using the same `comparison_col` target.
3. Run a forward pass on the validation set **without tracking gradients** (`torch.no_grad()`), since we're only doing inference, not training — this saves memory and compute.
4. Compute **AUC** (ranking quality, threshold-independent) and **F1** (precision/recall balance at the model's default decision threshold).


In [ ]:
# PASNet expects 32-bit float tensors throughout.
dtype = torch.FloatTensor

# Evaluation mode: disables dropout / training-only layer behaviour.
model.eval()

# --- Load data -----------------------------------------------------------
# x_train/y_train: used later as the SHAP "background" distribution.
# x_eval/y_eval:   the held-out validation cohort we evaluate the model on.
x_train, y_train = load_data(
    "../pipeline/output/pasnet/PASNet/Input/GO/Training.xlsx",
    dtype,
    comparison=comparison_col,
)

x_eval, y_eval = load_data(
    "../pipeline/output/pasnet/PASNet/Input/GO/Validation.xlsx",
    dtype,
    comparison=comparison_col,
)

# --- Inference -------------------------------------------------------------
# torch.no_grad() turns off autograd bookkeeping since we are not training.
with torch.no_grad():
    pred_new = model(x_eval)

# --- Metrics -----------------------------------------------------------------
auc_score = calc_auc(y_eval, pred_new)
f1_score_val = f1(y_eval, pred_new)

print("AUC on new dataset:", auc_score)
print("F1 on new dataset:", f1_score_val)


## 5. ROC Curve

The ROC (Receiver Operating Characteristic) curve plots the True Positive Rate against the False Positive Rate across every possible decision threshold, giving a fuller picture of discriminative ability than a single AUC number. We extract the model's softmax probability for the **positive class** (class index 1) and feed it, together with the true binary labels, into scikit-learn's `roc_curve`.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from sklearn.metrics import auc as sklearn_auc  # aliased to avoid clashing with calc_auc

# Re-run inference (kept separate/explicit so this cell is runnable on its own).
with torch.no_grad():
    pred_new = model(x_eval)

# Convert one-hot encoded labels back into a single class-index vector.
y_true = y_eval.argmax(dim=1).cpu().numpy()

# Convert raw model logits into calibrated probabilities for the positive class.
probs = torch.softmax(pred_new, dim=1)[:, 1].cpu().numpy()

# Compute the ROC curve coordinates and the area under it.
fpr, tpr, _ = roc_curve(y_true, probs)
roc_auc = sklearn_auc(fpr, tpr)

# --- Plot ------------------------------------------------------------------
plt.figure()
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], 'k--', label="Random")  # diagonal = chance-level baseline
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()


## 6. Export ROC Curve Coordinates

Save the raw `(FPR, TPR)` pairs to a tab-delimited text file so the curve can be reproduced or re-plotted later (e.g. in a manuscript figure) without re-running inference.


In [ ]:
# Persist the ROC curve's underlying coordinates for downstream re-plotting.
roc_output_path = "../pipeline/output/pasnet/PASNet/Output/GO/roc_curve_data.txt"

np.savetxt(
    roc_output_path,
    np.column_stack((fpr, tpr)),
    fmt="%.6f",
    header="FPR\tTPR",
    delimiter="\t",
)


## 7. Export Raw Model Predictions

Save the raw (pre-softmax) model outputs for every validation sample, so they can be re-analyzed later without needing to reload the model and data.


In [ ]:
# Save the raw per-class model outputs (logits) for every validation sample.
pred_output_path = "../pipeline/output/pasnet/PASNet/Output/GO/PASNet_pred.txt"
np.savetxt(pred_output_path, pred_new.numpy(), delimiter=",")


## 8. Explainable AI: Computing SHAP Values

PASNet's accuracy is only half the story — because it's built on biological pathways, we can also ask **which genes actually drove each prediction**. We use [SHAP](https://github.com/slundberg/shap) (SHapley Additive exPlanations), which assigns each input feature a contribution score for every individual prediction, grounded in cooperative game theory.

Specifically we use `shap.DeepExplainer`, designed for PyTorch/TensorFlow networks:
- It needs a **background dataset** (here, the training set) representing a baseline/reference distribution of feature values.
- It then estimates, for every validation sample, how much each gene's expression pushed the prediction up or down relative to that baseline.

We finish by computing the **mean absolute SHAP value** per gene (averaged over both samples and output classes) as a single global importance score, and save a ranked table of the most influential genes.


In [ ]:
# --- Compute per-sample, per-feature SHAP attributions ----------------------
# Background = training set (reference distribution for the explainer).
# Explained samples = validation set.
explainer = shap.DeepExplainer(model, x_train)
shap_values = explainer.shap_values(x_eval)

# --- Recover the human-readable gene/feature names --------------------------
# The raw training spreadsheet has an index column and the outcome label
# column mixed in with the actual gene features, so both are dropped here to
# leave only the columns the model actually consumed as input.
tr_dt = pd.read_excel("../pipeline/output/pasnet/PASNet/Input/GO/Training.xlsx")
tr_dt = tr_dt.drop([comparison_col], axis=1)
tr_dt = tr_dt.drop('Unnamed: 0', axis=1)

# --- Global feature importance ----------------------------------------------
# Average the absolute SHAP value first across samples (axis=1), then across
# output classes (axis=0), collapsing per-sample/per-class attributions into
# one global importance score per gene.
vals = np.abs(shap_values).mean(axis=1).mean(axis=0)
feature_importance = pd.DataFrame(list(zip(tr_dt.columns, vals)), columns=['Feature', 'Importance'])

# Rank genes from most to least influential.
feature_importance.sort_values(by='Importance', ascending=False, inplace=True)

# Persist the ranked table for downstream reporting / supplementary tables.
feature_importance.to_csv("../pipeline/output/pasnet/PASNet/Output/GO/feature_importance.csv", index=False)


## 9. SHAP Summary Plot

A SHAP *summary plot* shows, for the top-ranked genes, the distribution of each gene's SHAP value across all validation samples, colored by that gene's own expression level. This lets us see **both** how important a gene is *and* the direction of its effect — e.g. "high expression of gene X pushes predictions toward IGHV-mutated."

The steps below:
1. Reload the saved feature-importance ranking and pick out the corresponding columns/SHAP values, taking care to keep gene names, SHAP values, and expression values perfectly row/column-aligned.
2. Min-max scale the expression values purely for consistent **color** mapping (low = blue, high = red) on the plot.
3. Draw the SHAP dot plot with a custom diverging colormap, then export a high-resolution PNG suitable for a manuscript figure.


In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

# =====================================================================
# LOAD CSV AND EXTRACT FEATURES
# =====================================================================
# Re-load the ranked feature-importance table computed in the previous cell,
# so the plotting logic below can be re-run independently if needed.
csv_path = "../pipeline/output/pasnet/PASNet/Output/GO/feature_importance.csv"
df_importance = pd.read_csv(csv_path)

# Extract feature (gene) names in importance-ranked order.
features = df_importance['Feature'].tolist()

# Map each gene name back to its column index in the original training data,
# so we can pull matching columns out of the SHAP value matrix.
feature_indices = [tr_dt.columns.get_loc(col) for col in features]

# =====================================================================
# PROCESS DATA AND ALIGN SHAP VALUES (FIXED SHAPE ALIGNMENT)
# =====================================================================
# Normalize x_eval into a plain numpy array regardless of whether it's a
# torch tensor or already an array/DataFrame.
if hasattr(x_eval, 'cpu'):
    x_eval_np = x_eval.cpu().detach().numpy()
else:
    x_eval_np = x_eval

# DeepExplainer can return a list (one array per output class) or a single
# array; normalize to a single array here.
shap_plot_values = shap_values[0] if isinstance(shap_values, list) else shap_values

# CRITICAL FIX: Ensure row indices match identically between features and SHAP values
# (guards against any silent row-count or column-order mismatch before plotting).
if isinstance(x_eval, pd.DataFrame):
    x_eval_filtered = x_eval[features].copy()
    shap_plot_values_filtered = shap_plot_values[:len(x_eval_filtered), feature_indices]
else:
    num_samples = min(x_eval_np.shape[0], shap_plot_values.shape[0])
    x_eval_filtered = x_eval_np[:num_samples, feature_indices]
    shap_plot_values_filtered = shap_plot_values[:num_samples, feature_indices]

# Normalize expression data colors uniformly between 0 (Low) and 1 (High);
# this scaling is purely cosmetic for the colorbar, it does not affect the
# SHAP values themselves.
scaler = MinMaxScaler()
x_eval_scaled = scaler.fit_transform(x_eval_filtered)

# =====================================================================
# VISUALIZATION SETUP
# =====================================================================
plt.close('all')
fig, ax = plt.subplots(figsize=(12, 10))

coolwarm_cmap = plt.cm.coolwarm

# Generate the SHAP summary ("beeswarm") plot for the top-ranked features.
shap.summary_plot(
    shap_plot_values_filtered,
    x_eval_filtered,
    feature_names=features,
    plot_type="dot",
    max_display=20,        # show only the top 20 most important genes
    show=False,             # defer rendering so we can customize the figure below
    color_bar=False,        # we draw our own colorbar further down for finer control
    cmap=coolwarm_cmap,
    alpha=0.85,
)

# Apply standard linear scaling to X-axis
ax.set_xscale('linear')

# Define explicit linear ticks and math-text labels
ax.set_xticks([-0.1, 0, 0.1])
ax.set_xticklabels([r'$-0.1$', r'$0$', r'$0.1$'])

# =====================================================================
# COLORBAR CONFIGURATION
# =====================================================================
# Manually-built colorbar (Low -> High gene activity) replacing SHAP's default,
# for cleaner, publication-ready labeling.
m = plt.cm.ScalarMappable(cmap=coolwarm_cmap)
m.set_array([0, 1])
cb = fig.colorbar(m, ax=ax, ticks=[0, 1], aspect=50)
cb.set_ticklabels(['Low Activity', 'High Activity'])
cb.set_label('Gene Activity Level', weight='bold')

# =====================================================================
# PLOT STYLING & EXPORT
# =====================================================================
plt.title("SHAP Values", pad=15, size=13, weight='bold')
plt.xlabel("SHAP value (impact on model output)")
plt.tight_layout()

# Save the figure with high resolution before displaying it
plt.savefig('../pipeline/output/pasnet/PASNet/Output/GO/shap_values.png', dpi=300, bbox_inches='tight')

# Display the plot
plt.show()


## 10. Extract Pathway-Layer Weights

The defining feature of PASNet is its first sparse layer, conventionally named `sc1` ("sparse-coding layer 1") or `pathway_layer`, which connects individual genes to the biological pathways they belong to. Extracting this weight matrix lets us see, **independent of any single prediction**, how strongly each gene contributes to each pathway-level hidden unit inside the trained model.

Because the pickled `model` object could in principle be either (a) a live PyTorch model instance or (b) a plain state dictionary (depending on how it was saved upstream), the code below handles both cases defensively, with sensible fallbacks if the expected layer name isn't found.


In [ ]:
# --- Locate the sparse pathway layer's weight tensor -------------------------
if isinstance(model, dict):
    # Case A: `model` is a state dictionary / checkpoint dict rather than a
    # live nn.Module instance.
    state_dict = model.get('state_dict', model)

    # Search (case-insensitively) for a key that looks like the pathway layer.
    target_key = None
    for key in state_dict.keys():
        if 'sc1' in key.lower() or 'pathway' in key.lower():
            target_key = key
            break

    if target_key:
        weights = state_dict[target_key]
        print(f"Extracting array from dictionary key: '{target_key}'")
    else:
        # No layer matched by name — fall back to the first available tensor
        # so the cell still produces *something* rather than failing outright.
        target_key = list(state_dict.keys())[0]
        weights = state_dict[target_key]
        print(f"Target layer not matched. Defaulting to first key: '{target_key}'")

else:
    # Case B: `model` is a live PyTorch model instance — read the weight
    # tensor directly off the relevant submodule.
    if hasattr(model, 'sc1'):
        weights = model.sc1.weight
        print("Extracting array from model attribute: 'sc1'")
    elif hasattr(model, 'pathway_layer'):
        weights = model.pathway_layer.weight
        print("Extracting array from model attribute: 'pathway_layer'")
    else:
        # Neither expected attribute exists — fall back to the very first
        # parameter tensor in the model as a last resort.
        weights = next(model.parameters())
        print("Target attribute not matched. Extracting first parameter block.")

# --- Convert from PyTorch Tensor to NumPy array -----------------------------
if isinstance(weights, torch.Tensor):
    # detach() drops gradient tracking, cpu() ensures it's not still on a GPU.
    weights_np = weights.detach().cpu().numpy()
else:
    weights_np = np.array(weights)

# --- Save the structural (gene x pathway) weight matrix to CSV --------------
df = pd.DataFrame(weights_np)
output_path = "../pipeline/output/pasnet/PASNet/Output/GO/sc1_weights.xlsx"
df.to_excel(output_path, index=False)

print(f"Success! Extracted matrix shape {weights_np.shape} saved to {output_path}")


## Summary of Generated Outputs

Running this notebook end-to-end produces the following artifacts inside `pipeline/output/pasnet/PASNet/Output/GO/`:

| File | Contents |
|---|---|
| `roc_curve_data.txt` | Raw `(FPR, TPR)` coordinates of the ROC curve |
| `PASNet_pred.txt` | Raw per-class model logits for every validation sample |
| `feature_importance.csv` | Genes ranked by mean absolute SHAP value |
| `shap_values.png` | Publication-ready SHAP summary (beeswarm) plot |
| `sc1_weights.csv` | Gene → pathway weight matrix from PASNet's sparse first layer |

Together, these give both a **performance summary** (AUC/F1, ROC) and an **interpretability summary** (which genes/pathways the model relies on, and how) for the trained PASNet model.
